# Thin-walled U-channel cantilever: a shell formulation in LowLevelFEM.jl

A thin-walled channel provides a compact example of **bending, transverse shear and torsion** in a folded shell structure. The beam is represented by its three connected midsurfaces, with translational and rotational unknowns expressed in global Cartesian coordinates.

This tutorial develops a linear, small-displacement Reissner–Mindlin-type formulation for the planar shell facets using the weak-form DSL. It demonstrates how to:

- construct membrane, bending and transverse-shear operators explicitly;
- assemble their stiffness contributions with selective integration;
- couple the free-end cross-section to a remote reference point using a rigid rotation map and multipoint constraints (MPCs);
- recover stress resultants and in-plane stresses from the solution.

No dedicated U-channel element is required: the geometry, kinematics, constitutive matrices and constraints define the model.

In [187]:
using LowLevelFEM, LinearAlgebra#, SparseArrays

## 1. Geometry and mesh

Open the accompanying `U-beam_as_shell.geo` file in Gmsh. The default channel has an overall width of 50 mm, a height of 100 mm, a thickness of 5 mm and a length of 500 mm along the global $z$ axis. The geometry describes the web and flanges by their midsurfaces and meshes them with quadrilateral elements.

The physical groups identify the integration surface (`shell`), the clamped end (`support`) and the free-end contour (`load`). Mesh density and geometry parameters are exposed in the preprocessor. Close the preprocessor window to continue executing the notebook. The saved results below illustrate a refined mesh; the number of degrees of freedom and numerical values may change with mesh settings.

In [188]:
openGeometry("U-beam_as_shell.geo")
openPreProcessor()

## 2. Remote loading point

A point load applied to one shell node would introduce a local load concentration. Here the end contour is coupled to a single reference point so that the cross-section moves and rotates as a rigid body.

For this example, the reference point is the centroid of the free-end **midsurface contour**. With constant wall thickness this is the centroid of the thin-walled section idealization. Compute it by line integration over the group `load`:

$$
L_r=\int_{\Gamma_{\mathrm{load}}}1\,\mathrm ds,
\qquad
\mathbf r_c=\frac{1}{L_r}\int_{\Gamma_{\mathrm{load}}}\mathbf r\,\mathrm ds.
$$

The variable `L` below is the contour length, not the cantilever length. The point is added as the physical point `remote`, and the mesh is regenerated to include its degrees of freedom. The commented alternative location is inactive; the applied force acts at the contour centroid.

In [189]:
surf = "shell"
mat = Material(surf)
U0 = Field([mat], type=:VectorField, dim=3)

L = ∫(U0, "load", (x,y,z)->1)
xc = ∫(U0, "load", (x,y,z)->x) / L
yc = ∫(U0, "load", (x,y,z)->y) / L
zc = ∫(U0, "load", (x,y,z)->z) / L;

In [190]:
#xc = -11.497640687336565

In [191]:
Premote = gmsh.model.occ.addPoint(xc, yc, zc)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
#gmsh.model.mesh.setOrder(3)
#gmsh.model.mesh.renumberNodes()
gmsh.model.mesh.generate(2)

## 3. Material and independent fields

Use isotropic linear elasticity with $E=200\,000$ N/mm² and $\nu=0.3$. Define two independent vector-valued field descriptors:

- `U`: three Cartesian components of midsurface displacement $\mathbf u$;
- `Φ`: three Cartesian components of the infinitesimal rotation vector $\boldsymbol\varphi$.

Both fields are defined in the three-dimensional ambient space (`dim=3`), even though their physical stiffness is integrated over two-dimensional surfaces. Both use the mesh interpolation order. Including `remote` in their material lists creates translational and rotational degrees of freedom at the isolated reference point; the stiffness itself is assembled only on `shell`.

In [192]:
mat = Material(surf, E=2e5, ν=0.3)
rem = Material("remote")

U = Field([mat,rem], type=:VectorField, dim=3, field=:u)
Φ = Field([mat,rem], type=:VectorField, dim=3, field=:φ, reducedOrder=false);

## 4. Local shell frame

For each planar facet, construct a right-handed local frame $(\mathbf t_\xi,\mathbf t_\eta,\mathbf n)$, where $\mathbf n$ is the shell normal and $\mathbf t_\xi,\mathbf t_\eta$ are tangent directions. LowLevelFEM returns these directions as fields, so their components may vary from one facet to another.

The global translational and rotational fields can therefore remain continuous along the fold lines while the local shell strain components are evaluated in each facet's own coordinate system. Component arrays are extracted for use in the matrix chains.

In [193]:
n = normalVector(U, surf)
tξ, tη = tangentVectors(U, surf);

In [194]:
n = [n[1], n[2], n[3]]
tξ = [tξ[1], tξ[2], tξ[3]]
tη = [tη[1], tη[2], tη[3]];

## 5. Reissner–Mindlin-type kinematics

For a flat facet, let $\zeta$ be the signed thickness coordinate measured from the midsurface. The linear displacement ansatz is

$$
\mathbf u(\xi,\eta,\zeta)=\mathbf u(\xi,\eta)
+\zeta\,\boldsymbol\varphi(\xi,\eta)\times\mathbf n.
$$

Write $\mathbf u_{,\xi}=(\nabla\mathbf u)\mathbf t_\xi$, and define the other directional derivatives analogously. The membrane strain and curvature vectors are

$$
\boldsymbol\varepsilon_m=\begin{bmatrix}
\mathbf t_\xi\cdot\mathbf u_{,\xi}\\
\mathbf t_\eta\cdot\mathbf u_{,\eta}\\
\mathbf t_\xi\cdot\mathbf u_{,\eta}+\mathbf t_\eta\cdot\mathbf u_{,\xi}
\end{bmatrix},
\qquad
\boldsymbol\kappa_b=\begin{bmatrix}
\mathbf t_\eta\cdot\boldsymbol\varphi_{,\xi}\\
-\mathbf t_\xi\cdot\boldsymbol\varphi_{,\eta}\\
\mathbf t_\eta\cdot\boldsymbol\varphi_{,\eta}-\mathbf t_\xi\cdot\boldsymbol\varphi_{,\xi}
\end{bmatrix}.
$$

The transverse-shear components are

$$
\gamma_\eta=\mathbf n\cdot\mathbf u_{,\eta}-\mathbf t_\xi\cdot\boldsymbol\varphi,
\qquad
\gamma_\xi=\mathbf n\cdot\mathbf u_{,\xi}+\mathbf t_\eta\cdot\boldsymbol\varphi.
$$

The generalized strain vector is stored in the order $[\boldsymbol\varepsilon_m^{\mathsf T},\boldsymbol\kappa_b^{\mathsf T},\gamma_\eta,\gamma_\xi]^{\mathsf T}$. The Kronecker products in `A_u` and `A_φ` select the directional derivatives from `Grad(U)` and `Grad(Φ)`, while `G_φ` supplies the undifferentiated rotations in the shear strains. The third entries of the membrane and curvature vectors use engineering shear conventions.

In [195]:
# kron(tξ, tη)' ⋅ Grad(U) gives tξᵀ(∇U)tη.

A_u = vcat(kron(tξ,tξ)',
           kron(tη,tη)',
           (kron(tξ,tη) + kron(tη,tξ))',
           zeros(3,9),
           kron(n,tη)',
           kron(n,tξ)')

A_φ = vcat(zeros(3,9),
           kron(tη,tξ)',
           -kron(tξ,tη)',
           (kron(tη,tη) - kron(tξ,tξ))',
           zeros(2,9))

G_φ = vcat(zeros(6,3), -tξ', tη');

## 6. Constitutive matrices and weak form

Let $b$ denote the wall thickness in the Julia code (the geometry parameter `t`). For a homogeneous isotropic plate with plane-stress constitutive matrix $\mathbf C$, integrating over the symmetric thickness gives

$$
\mathbf D_1=b\mathbf C,\qquad
\mathbf D_2=\frac{b^3}{12}\mathbf C,\qquad
\mathbf D_3=k_s b\mu\mathbf I_2,
$$

with shear modulus $\mu=E/[2(1+\nu)]$ and the shear correction factor $k_s=4/5$ used in this example. There is no membrane–bending coupling for this symmetric homogeneous thickness distribution.

The internal virtual work is

$$
\begin{aligned}
\delta W_{\mathrm{int}}={}&\int_{\Gamma_s}
\left(\delta\boldsymbol\varepsilon_m^{\mathsf T}\mathbf D_1\boldsymbol\varepsilon_m
+\delta\boldsymbol\kappa_b^{\mathsf T}\mathbf D_2\boldsymbol\kappa_b\right)\,\mathrm d\Gamma\\
&+\int_{\Gamma_s}\delta\boldsymbol\gamma^{\mathsf T}\mathbf D_3\boldsymbol\gamma\,\mathrm d\Gamma.
\end{aligned}
$$

In the DSL, `B_mb` combines the membrane and bending rows, and `B_s` contains the two shear rows. The following two matrix chains mirror the two integrals. Use full quadrature for membrane and bending, and reduced quadrature only for transverse shear to alleviate shear locking. The keyword `Γ=surf` selects the shell surfaces in the three-dimensional model.

In [196]:
b = getParameter("t")
k_s = 4/5

C = D(:PlaneStress, mat)

D1 = b*C

D2 = b^3 / 12 * C

D3 = k_s * b * mat.μ * Matrix{Float64}(I, 2, 2)

# Locking miatt a nyírást csak redukáltan integráljuk.
B_mb = A_u[1:6, :] ⋅ Grad(U) + A_φ[1:6, :] ⋅ Grad(Φ)
B_s  = A_u[7:8, :] ⋅ Grad(U) + G_φ[7:8, :] ⋅ Φ
D_mb = cat(D1, D2; dims=(1, 2))

K = ∫(B_mb' ⋅ D_mb ⋅ B_mb; Γ=surf) +
    ∫(B_s'  ⋅ D3   ⋅ B_s;  Γ=surf, gauss=:reduced)

K[:,:]

101916×101916 SparseArrays.SparseMatrixCSC{Float64, Int64} with 3488954 stored entries:
⎡⣿⣿⣻⣟⣛⠛⠯⠭⠿⠭⠭⠭⠭⠭⠽⠯⠿⠭⠭⠭⣿⣟⣿⣛⡛⠛⠯⠽⠯⠭⠭⠭⠭⠭⠿⠯⠿⠭⠭⠭⎤
⎢⣿⢾⡻⣮⡉⠛⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣮⢷⣮⡉⠛⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣧⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠸⡆⠙⢿⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡏⡇⠉⠀⠀⠈⠻⣦⡉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⣿⠀⠁⠀⠀⠉⠳⣄⠙⠛⠶⢦⣤⣀⡀⠀⠀⠀⠀⠀⎥
⎢⡟⡇⠀⠀⠀⠀⣧⠈⠻⣦⡀⠀⠀⠈⠉⠁⠀⠀⠀⠀⣿⠀⠀⠀⠀⠸⣇⠈⢿⣦⡀⠀⠀⠉⠉⠁⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠸⡆⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⢻⡄⠀⠙⢿⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠀⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠈⣷⠀⠀⠀⠙⢿⣦⡀⠀⠀⠀⠀⠀⎥
⎢⡷⡇⠀⠀⠀⠀⠀⠈⠇⠀⠀⠀⠀⠈⠻⣦⣤⣄⣀⠀⡿⡄⠀⠀⠀⠀⠀⠸⠇⠀⠀⠀⠀⠙⠿⣆⣤⣄⣀⠀⎥
⎢⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⠻⣦⡉⠉⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣯⢿⣮⡉⠉⎥
⎢⡇⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠈⠻⣦⣧⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡆⠙⢿⣦⎥
⎢⣿⢿⡻⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠯⠿⠭⠭⠿⣿⣿⣿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠯⠿⠭⠭⠿⎥
⎢⣿⢻⡹⣷⣌⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⠻⣦⡉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠈⠻⣷⡄⠀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠸⡇⠈⠻⣦⡀⠀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣏⡇⠀⠀⠀⠀⠙⢦⡉⠙⠛⠶⢦⣤⣀⡀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠈⠻⣦⡉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⎥
⎢⡏⡇⠀⠀⠀⠀⣷⠀⠻⣷⣄⠀⠀⠀⠉⠁⠀⠀⠀⠀⣿⠀⠀⠀⠀⠘⣇⠈⠻⣦⡀⠀⠀⠀⠉⠁⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠸⣇⠀⠈⠻⣷⣄⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⢹⡄⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠀⢻⡄⠀⠀⠈⠻⣷⣄⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⡿⡇⠀⠀⠀⠀⠀⠈⠇⠀⠀⠀⠀⠈⠻⢧⡤⣤⣄⣀⡿⡄⠀⠀⠀⠀⠀⠘⠇⠀⠀⠀⠀⠈⠻⣦⡤⣤⣀⡀⎥
⎢⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⡻⣷⣌⠉⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣯⠻⣦⡈⠉⎥
⎣⡇⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠈⠻⣷⣧⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⡆⠈⠻⣦⎦

## 7. Drilling rotation stabilization

The rotation about the local normal, $\boldsymbol\varphi\cdot\mathbf n$, is not independently constrained by the membrane, bending and transverse-shear terms on a planar facet. A small drilling penalty ties it to the in-plane infinitesimal rotation of the midsurface,

$$
\omega_n=\frac12\left(\mathbf t_\eta\cdot\mathbf u_{,\xi}-\mathbf t_\xi\cdot\mathbf u_{,\eta}\right),
\qquad
\gamma_d=\mathbf n\cdot\boldsymbol\varphi-\omega_n.
$$

Add the weak-form contribution $\int_{\Gamma_s}\delta\gamma_d\,\alpha b\mu\,\gamma_d\,\mathrm d\Gamma$ with $\alpha=10^{-4}$. This is a numerical stabilization for the drilling degree of freedom, not an additional physical torsion law.

In [197]:
A_d = (kron(tξ,tη) - kron(tη,tξ))' / 2
B_d = A_d ⋅ Grad(U) + n' ⋅ Φ

α = 1e-4
μ = mat.μ
Kd = ∫(B_d' ⋅ (α * b * μ) ⋅ B_d; Γ=surf)
Kd[:,:]

101916×101916 SparseArrays.SparseMatrixCSC{Float64, Int64} with 2414062 stored entries:
⎡⣿⣿⣻⣟⣛⠛⠯⠭⠿⠭⠭⠭⠭⠭⠽⠯⠿⠭⠭⠭⣿⣟⣿⣛⡛⠛⠯⠽⠯⠭⠭⠭⠭⠭⠿⠯⠿⠭⠭⠭⎤
⎢⣿⢾⡻⣮⡉⠛⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣮⢷⣮⡉⠛⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⣧⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠸⡆⠙⢿⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡏⡇⠉⠀⠀⠈⠻⣦⡉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⣿⠀⠁⠀⠀⠉⠳⣄⠙⠛⠶⢦⣤⣀⡀⠀⠀⠀⠀⠀⎥
⎢⡟⡇⠀⠀⠀⠀⣧⠈⠻⣦⡀⠀⠀⠈⠉⠁⠀⠀⠀⠀⣿⠀⠀⠀⠀⠸⣇⠈⢿⣦⡀⠀⠀⠉⠉⠁⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠸⡆⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⢻⡄⠀⠙⢿⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠀⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠈⣷⠀⠀⠀⠙⢿⣦⡀⠀⠀⠀⠀⠀⎥
⎢⡷⡇⠀⠀⠀⠀⠀⠈⠇⠀⠀⠀⠀⠈⠻⣦⣤⣄⣀⠀⡿⡄⠀⠀⠀⠀⠀⠸⠇⠀⠀⠀⠀⠙⠿⣆⣤⣄⣀⠀⎥
⎢⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⠻⣦⡉⠉⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣯⢿⣮⡉⠉⎥
⎢⡇⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠈⠻⣦⣧⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡆⠙⢿⣦⎥
⎢⣿⢿⡻⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠯⠿⠭⠭⠿⣿⣿⣿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠯⠿⠭⠭⠿⎥
⎢⣿⢻⡹⣷⣌⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣿⠻⣦⡉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠈⠻⣷⡄⠀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠸⡇⠈⠻⣦⡀⠀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣏⡇⠀⠀⠀⠀⠙⢦⡉⠙⠛⠶⢦⣤⣀⡀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠈⠻⣦⡉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⎥
⎢⡏⡇⠀⠀⠀⠀⣷⠀⠻⣷⣄⠀⠀⠀⠉⠁⠀⠀⠀⠀⣿⠀⠀⠀⠀⠘⣇⠈⠻⣦⡀⠀⠀⠀⠉⠁⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠸⣇⠀⠈⠻⣷⣄⠀⠀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⢹⡄⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡇⡇⠀⠀⠀⠀⠀⢻⡄⠀⠀⠈⠻⣷⣄⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⡿⡇⠀⠀⠀⠀⠀⠈⠇⠀⠀⠀⠀⠈⠻⢧⡤⣤⣄⣀⡿⡄⠀⠀⠀⠀⠀⠘⠇⠀⠀⠀⠀⠈⠻⣦⡤⣤⣀⡀⎥
⎢⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⡻⣷⣌⠉⡟⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣯⠻⣦⡈⠉⎥
⎣⡇⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡇⠈⠻⣷⣧⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⡆⠈⠻⣦⎦

## 8. Rigid end cross-section and MPCs

At the free end, tie the auxiliary translations and the rotations of all nodes on `load` to the corresponding degrees of freedom at `remote`. A pure translation constraint alone would prevent the correct motion of an offset point during rigid rotation. For a slave node $i$ and master point $m$, the physical displacement must satisfy

$$
\mathbf u_i=\mathbf u_m+\boldsymbol\varphi_m\times(\mathbf r_i-\mathbf r_m).
$$

The rotation map `R` implements the second term. It acts on the rotation field, while the MPCs impose equality of the auxiliary translations and rotations. The construction allows the free-end contour to translate and rotate as a rigid cross-section and restrains its independent warping.

With auxiliary displacement $\mathbf u_0$, the physical fields are related by

$$
\begin{bmatrix}\mathbf u\\\boldsymbol\varphi\end{bmatrix}
=\underbrace{\begin{bmatrix}\mathbf I&\mathbf R\\\mathbf 0&\mathbf I\end{bmatrix}}_{\mathbf T}
\begin{bmatrix}\mathbf u_0\\\boldsymbol\varphi\end{bmatrix}.
$$

For the original stiffness $\mathbf K_0=\mathbf K+\mathbf K_d$, virtual work therefore requires $\mathbf K_1=\mathbf T^{\mathsf T}\mathbf K_0\mathbf T$. The corresponding load transformation will be applied before solving.

In [198]:
mpc_u = MultiPointConstraint(master="remote", slave="load", field=U)
mpc_φ = MultiPointConstraint(master="remote", slave="load", field=Φ)

R = rigidRotationMap(mpc_u, mpc_φ);

In [199]:
K0 = K + Kd
T = rigidRotationTransformation(K0, R)
K1 = T' * K0 * T

sparse([1, 3, 28, 30, 55, 57, 253, 255, 571, 573  …  53989, 53991, 92635, 92637, 101272, 101274, 101908, 101910, 101914, 101916], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  101916, 101916, 101916, 101916, 101916, 101916, 101916, 101916, 101916, 101916], [457966.4017664732, -178569.02472527488, 17638.38904203577, -4577.953296703507, -323560.3915029541, 18311.81318681368, -66706.21780272941, 4577.95329670331, 29153.41893879256, -18311.813186813375  …  9.022187441587448e-10, -20351.6085789692, 661375.6613756518, -460878.193633597, 1.6552803572267294e-8, -2.5762849858357324e6, 9.851646609604359e-9, -20351.608579312742, -1.0710209608078003e-8, 1.0072172176320411e7], 101916, 101916)

### Equivalent block-matrix construction

The same transformation can be written explicitly in terms of the displacement and rotation blocks of $\mathbf K_0$:

$$
\mathbf T^{\mathsf T}\mathbf K_0\mathbf T=
\begin{bmatrix}
\mathbf K_{uu} & \mathbf K_{u\varphi}+\mathbf K_{uu}\mathbf R\\
\mathbf K_{\varphi u}+\mathbf R^{\mathsf T}\mathbf K_{uu} &
\mathbf K_{\varphi\varphi}+\mathbf K_{\varphi u}\mathbf R+\mathbf R^{\mathsf T}\mathbf K_{u\varphi}+\mathbf R^{\mathsf T}\mathbf K_{uu}\mathbf R
\end{bmatrix}.
$$

The next cell builds this alternative directly with `SystemMatrix` block operations and compares it to `K1`. The saved relative matrix difference is about $4.65\times10^{-17}$, consistent with floating-point roundoff. Both constructions describe the same rigid-rotation transformation; the subsequent solve uses `K1`.

In [200]:
K0 = K + Kd
nu = size(R, 1)
iu = 1:nu
iφ = (nu+1):size(K0, 1)

Kuu = SystemMatrix(K0[iu, iu], U)
Kuφ = SystemMatrix(K0[iu, iφ], Φ, U)
Kφu = SystemMatrix(K0[iφ, iu], U, Φ)
Kφφ = SystemMatrix(K0[iφ, iφ], Φ)

K2 = SystemMatrix([
    Kuu              Kuφ + Kuu*R
    Kφu + R'*Kuu     Kφφ + Kφu*R + R'*Kuφ + R'*Kuu*R
])

norm(K1.A - K2.A) / norm(K1.A)

4.645595555883995e-17

## 9. Apply loads and solve the constrained system

Apply a 1000 N force in the global $y$ direction at `remote`, with zero applied reference-point moment. Assemble both force and moment fields into the block `SystemVector`. Transform the load consistently with the stiffness: $\mathbf f_1=\mathbf T^{\mathsf T}\mathbf f$.

At `support`, constrain all three translations and all three rotations to zero. Solve the transformed system with the two MPC definitions and the clamped boundary conditions. The solution returns the auxiliary displacement `u0` and the rotation field `φ` in block order.

In [201]:
fu = ∫(U ⋅ [0.0, 1000.0, 0.0]; Γ="remote")
fφ = ∫(Φ ⋅ [0.0, 0.0, 0.0]; Γ="remote")
f = SystemVector([fu, fφ]);
f1 = T' * f

bc_u = BoundaryCondition("support"; field=U, ux=0, uy=0, uz=0)
bc_φ = BoundaryCondition("support"; field=Φ, φx=0, φy=0, φz=0);

In [202]:
showDoFResults(fu, name="F")
showDoFResults(fφ, name="M");

In [203]:
u0, φ = solveField(Symmetric(K1), f1, support=[bc_u, bc_φ], mpc=[mpc_u, mpc_φ]);

## 10. Recover the physical displacement

Recover the physical displacement with the same map used during assembly:

$$
\mathbf u=\mathbf u_0+\mathbf R\boldsymbol\varphi.
$$

Restrict the elementwise result to the shell surface so that auxiliary point degrees of freedom do not enter the shell postprocessing. Display the recovered displacement field with a deformation scale of 10.

In [204]:
u = nodesToElements(u0 + R * φ, onPhysicalGroup=surf);

In [205]:
showDoFResults(u, name="u", visible=true, factor=10);

## 11. Recover membrane, bending and transverse-shear resultants

Evaluate the field gradients of the physical displacement and rotation, then reconstruct the same local strains used in the weak form. The membrane, bending and transverse-shear resultants follow directly:

$$
\mathbf N=\mathbf D_1\boldsymbol\varepsilon_m,
\qquad
\mathbf M=\mathbf D_2\boldsymbol\kappa_b,
\qquad
\begin{bmatrix}Q_\eta\\Q_\xi\end{bmatrix}
=\mathbf D_3\begin{bmatrix}\gamma_\eta\\\gamma_\xi\end{bmatrix}.
$$

Here $\mathbf N=[N_\xi,N_\eta,N_{\xi\eta}]^{\mathsf T}$ is the in-plane force per unit length, $\mathbf M=[M_\xi,M_\eta,M_{\xi\eta}]^{\mathsf T}$ is the bending/twisting moment per unit length, and $Q_\xi,Q_\eta$ are transverse-shear forces per unit length. With the N–mm unit system, $\mathbf N$ and $\mathbf Q$ have units N/mm, while $\mathbf M$ has units N·mm/mm.

The computed fields retain element-local values. For the affine Q1 quadrilaterals used here, the nodal mean gives the element-center shear value, consistent with the reduced one-point shear integration. The helper `center` stores that value at every local node for visualization.

In [206]:
n = normalVector(U, surf)
tξ, tη = tangentVectors(U, surf)
φe = nodesToElements(φ, onPhysicalGroup=surf)

gu = ∇(u)
gφ = ∇(φe)
uξ, uη = gu * tξ, gu * tη
φξ, φη = gφ * tξ, gφ * tη

εm = [tξ ⋅ uξ, tη ⋅ uη, (tξ ⋅ uη) + (tη ⋅ uξ)]
κb = [tη ⋅ φξ, -(tξ ⋅ φη), (tη ⋅ φη) - (tξ ⋅ φξ)]

Nξ, Nη, Nξη = D1 * εm
Mξ, Mη, Mξη = D2 * κb

γξ = (n ⋅ uξ) + (φe ⋅ tη)
γη = (n ⋅ uη) - (φe ⋅ tξ)

# On this affine Q1 mesh, the nodal mean gives the element-center value.
center(s) = ScalarField(
    [repeat(sum(a; dims=1) / size(a, 1), size(a, 1), 1) for a in s.A],
    [;;], s.t, s.numElem, s.nsteps, s.type, s.model
)

Qη, Qξ = D3 * [center(γη), center(γξ)];

### Inspect the free-end membrane shear

Transfer the local membrane shear resultant $N_{\xi\eta}$ to the `load` contour with `elementsToElements`, preserving elementwise data. The tangent field `e` supplies the direction along the contour. The following diagnostic integrates a moment about the global $z$ axis and a force component along $y$ using the indicated sign convention.

The ratio `xT = Ms / Fy` is a moment-to-force length obtained from this particular membrane-shear contribution in the already loaded shell. It is **not** an independent determination of the cross-section's shear center: the end solution also includes torsion, restrained warping and other shell resultants. This diagnostic does not change the location of the remote point or the applied load.

In [207]:
r = VectorField(U, "load", [(x,y,z)->x, (x,y,z)->y, (x,y,z)->z])
Nxy = elementsToElements(Nξη, onPhysicalGroup="load")
e = tangentVector(U, "load")
ez = VectorField(U, "load", [0,0,1])
ey = VectorField(U, "load", [0,1,0])

showDoFResults(e, name="e")
plotOnBeam("load", Nxy, name="Nξη at the end")

Ms = ∫(U, "load", (r × (-Nxy * e)) * ez)

Fy = ∫(U, "load", (Nxy * e) * ey)

xT = Ms / Fy

-12.004635383514508

### Visualize the local frames and resultants

Displaying the local normals and tangents helps verify the sign conventions on the web and flanges. Display each resultant as an element field in Gmsh to inspect its distribution along the cantilever and across the folds.

In [208]:
showElementResults(n, name="n")
showElementResults(tξ, name="tξ")
showElementResults(tη, name="tη")

7

In [209]:
showElementResults(Nξ;  name="N_xi")
showElementResults(Nη;  name="N_eta")
showElementResults(Nξη; name="N_xi_eta")

showElementResults(Mξ;  name="M_xi")
showElementResults(Mη;  name="M_eta")
showElementResults(Mξη; name="M_xi_eta")

showElementResults(Qξ;  name="Q_xi")
showElementResults(Qη;  name="Q_eta");

## 12. In-plane stress recovery through the thickness

With $\zeta$ measured from the midsurface along the local normal, the linear in-plane stress reconstruction is

$$
\boldsymbol\sigma(\zeta)=\frac{\mathbf N}{b}+\frac{12\zeta}{b^3}\,\mathbf M.
$$

Evaluate this expression at the top ($\zeta=+b/2$), middle ($\zeta=0$) and bottom ($\zeta=-b/2$) surfaces. For the in-plane stress components $[\sigma_\xi,\sigma_\eta,\tau_{\xi\eta}]$, the plane-stress von Mises equivalent stress is

$$
\sigma_{\mathrm{VM}}=\sqrt{\sigma_\xi^2-\sigma_\xi\sigma_\eta+\sigma_\eta^2+3\tau_{\xi\eta}^2}.
$$

The three plots show the in-plane equivalent stress in N/mm². The transverse-shear stresses are not included in this expression. Top and bottom refer to each facet's local normal orientation.

In [210]:
N = [Nξ, Nη, Nξη]
M = [Mξ, Mη, Mξη]

σ_top = N / b + (6 / b^2) * M
σ_mid = N / b
σ_bot = N / b - (6 / b^2) * M

vonMises(σ) = sqrt(σ[1]^2 - σ[1] * σ[2] + σ[2]^2 + 3 * σ[3]^2)

showElementResults(vonMises(σ_top); name="In-plane von Mises - top (+n)")
showElementResults(vonMises(σ_mid); name="In-plane von Mises - midsurface")
showElementResults(vonMises(σ_bot); name="In-plane von Mises - bottom (-n)");

## 13. Reference-point response

Probe the reference-point translation and rotation. Since `R` acts on the slave nodes and not the reference point itself, `u0` gives the physical translation at `remote`. The rotation is reported in radians about the three global axes.

In [211]:
probe(u0, "remote")

3-element Vector{Float64}:
 -0.0
  0.2672045089
  0.0

In [212]:
probe(φ, "remote")

3-element Vector{Float64}:
 -0.0004359395
 -0.0
  0.0036655029

### Interpretation of the results

The saved solution gives the following reference-point response:

| Quantity | Value | Unit |
|:--|--:|:--|
| $u_y$ | 0.267205 | mm |
| $\varphi_x$ | −0.00043594 | rad |
| $\varphi_z$ | 0.00366550 | rad |

The nonzero rotation about $z$ is physically significant. An open U-channel generally has a shear center outside its material cross-section, separate from its centroid. A transverse force applied at the centroid therefore causes both bending and torsion even when the applied moment at the reference point is zero. The clamped root and rigid loaded end also restrain cross-sectional warping. Consequently, this centroid-loaded shell response should not be compared directly with an uncoupled bending-only beam calculation.

The example illustrates how shell kinematics, numerical integration and rigid-end coupling can be expressed as readable weak forms and explicit transformations. Open the postprocessor to explore the displacement, resultants and stress views.

In [ ]:
openPostProcessor()